# 🚀 Day 6: Grover's Search Algorithm

## 14-Day Quantum DevRel Bootcamp

**Goal:** Implement and visualize Grover's search algorithm — your first quantum algorithm with a provable speedup.

**Today's Deliverables:**
1. ✅ Oracle construction for marked states
2. ✅ Diffusion operator (inversion about the mean)
3. ✅ Complete Grover's algorithm for 2, 3, and 4 qubits
4. ✅ Amplitude amplification visualization step by step
5. ✅ Success probability vs iteration count analysis
6. ✅ Classical vs quantum query comparison

**Key insight:** Grover's algorithm is a rotation in a 2D subspace. The oracle marks the target (phase flip), the diffusion operator reflects about the mean — two reflections = one rotation. After $\pi\sqrt{N}/4$ rotations, you land on the target with near-certainty.

---

## Section 1: The Oracle — Marking the Target

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

print("✅ All imports loaded!")
print()

# ============================================================
# Section 1: Oracle Construction
# ============================================================
# The oracle flips the phase of marked states:
#   O|w⟩ = -|w⟩  (marked)
#   O|x⟩ = |x⟩   (unmarked)

def build_oracle_matrix(n_qubits, marked_states):
    """Build the phase oracle as a diagonal matrix."""
    N = 2 ** n_qubits
    oracle = np.eye(N, dtype=complex)
    for s in marked_states:
        oracle[s, s] = -1
    return oracle

def build_oracle_circuit(n_qubits, marked_states):
    """Build oracle as a QuantumCircuit."""
    qc = QuantumCircuit(n_qubits)
    for target in marked_states:
        for i in range(n_qubits):
            if not (target >> i) & 1:
                qc.x(i)
        if n_qubits == 1:
            qc.z(0)
        else:
            qc.h(n_qubits - 1)
            qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
            qc.h(n_qubits - 1)
        for i in range(n_qubits):
            if not (target >> i) & 1:
                qc.x(i)
    return qc

# Demo: oracles
print("🔮 Oracle Construction")
print("=" * 55)
print()

# 2-qubit oracle for |11⟩
oracle_mat = build_oracle_matrix(2, [3])
print("Oracle matrix for 2-qubit search, target |11⟩:")
print(f"  diag = {np.diag(oracle_mat).real}")
print(f"  |00⟩ → +1, |01⟩ → +1, |10⟩ → +1, |11⟩ → -1  ✓")
print()

# Verify circuit matches matrix
oracle_circ = build_oracle_circuit(2, [3])
op = Operator(oracle_circ)
target_op = Operator(oracle_mat)
print(f"Circuit matches matrix: {op.equiv(target_op)}")
print()
print("Oracle circuit for |11⟩:")
print(oracle_circ.draw())
print()

# 3-qubit oracle for |101⟩
oracle3 = build_oracle_matrix(3, [5])
print("Oracle for 3-qubit search, target |101⟩ (index 5):")
diag3 = np.diag(oracle3).real
for i in range(8):
    marker = " ← target" if i == 5 else ""
    print(f"  |{i:03b}⟩: {diag3[i]:+.0f}{marker}")

---
## Section 2: The Diffusion Operator

The diffusion operator reflects about the uniform superposition $|s\rangle = H^{\otimes n}|0\rangle^n$:

$$D = 2|s\rangle\langle s| - I$$

This is also called **inversion about the mean**: states with amplitude above the mean get boosted, those below get suppressed.

### Circuit Implementation

| Step | Operation | Purpose |
| :--- | :--- | :--- |
| 1 | $H^{\otimes n}$ | Map $\vert s\rangle \to \vert 0\rangle^n$ |
| 2 | $X^{\otimes n}$ | Prepare for multi-controlled Z |
| 3 | MCZ | Phase flip everything except $\vert 0\rangle$ |
| 4 | $X^{\otimes n}$ | Undo step 2 |
| 5 | $H^{\otimes n}$ | Map back to original basis |

In [ ]:
# ============================================================
# Section 2: Diffusion Operator
# ============================================================

def build_diffusion_matrix(n_qubits):
    """D = 2|s⟩⟨s| - I"""
    N = 2 ** n_qubits
    s = np.ones(N, dtype=complex) / np.sqrt(N)
    return 2 * np.outer(s, s) - np.eye(N, dtype=complex)

def build_diffusion_circuit(n_qubits):
    """Circuit implementation of D."""
    qc = QuantumCircuit(n_qubits)
    qc.h(range(n_qubits))
    qc.x(range(n_qubits))
    qc.h(n_qubits - 1)
    qc.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    qc.h(n_qubits - 1)
    qc.x(range(n_qubits))
    qc.h(range(n_qubits))
    return qc

print("🔄 Diffusion Operator")
print("=" * 55)
print()

# Verify matrix
D2 = build_diffusion_matrix(2)
print("Diffusion matrix (2 qubits):")
print(np.round(D2.real, 3))
print()

# Verify circuit matches matrix
diff_circ = build_diffusion_circuit(2)
op_d = Operator(diff_circ)
target_d = Operator(D2)
print(f"Circuit matches matrix: {op_d.equiv(target_d)}")
print()

# Show what diffusion does: inversion about the mean
print("Inversion about the mean demonstration:")
# Start with non-uniform amplitudes (simulating post-oracle state)
amps = np.array([0.5, 0.5, 0.5, -0.5])  # |11⟩ phase-flipped
mean = np.mean(amps)
reflected = 2 * mean - amps
print(f"  Before diffusion: {amps}")
print(f"  Mean amplitude:   {mean:.4f}")
print(f"  After diffusion:  {reflected}")
print(f"  |11⟩ amplitude went from {amps[3]:.2f} → {reflected[3]:.2f}")
print()
print("💡 The marked state's amplitude is boosted from -0.5 to 1.0!")
print("   This is exactly how Grover amplifies the target.")

---
## Section 3: Complete Grover's Algorithm

The full algorithm:

1. **Initialize:** $|\psi_0\rangle = H^{\otimes n}|0\rangle^n$ (uniform superposition)
2. **Repeat** $k$ times:
   - Apply oracle $O_f$ (phase flip marked states)
   - Apply diffusion $D$ (inversion about the mean)
3. **Measure** all qubits

**Optimal iterations:** $k_{\text{opt}} \approx \frac{\pi}{4}\sqrt{N/M}$ where $M$ = number of marked states

In [ ]:
# ============================================================
# Section 3: Complete Grover's Algorithm
# ============================================================

def optimal_iterations(n_qubits, n_marked=1):
    """Optimal number of Grover iterations."""
    N = 2 ** n_qubits
    theta = np.arcsin(np.sqrt(n_marked / N))
    k = round(np.pi / (4 * theta) - 0.5)
    return max(1, k)

def grover_circuit(n_qubits, marked_states, n_iterations=None):
    """Build the complete Grover circuit."""
    if n_iterations is None:
        n_iterations = optimal_iterations(n_qubits, len(marked_states))
    
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.h(range(n_qubits))
    
    oracle = build_oracle_circuit(n_qubits, marked_states)
    diffusion = build_diffusion_circuit(n_qubits)
    
    for _ in range(n_iterations):
        qc.compose(oracle, inplace=True)
        qc.barrier()
        qc.compose(diffusion, inplace=True)
        qc.barrier()
    
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def run_grover(n_qubits, marked_states, n_iterations=None, shots=1024):
    """Run Grover's algorithm and return measurement counts."""
    qc = grover_circuit(n_qubits, marked_states, n_iterations)
    sim = AerSimulator()
    result = sim.run(qc, shots=shots).result()
    return result.get_counts()

print("🔍 Grover's Algorithm — Full Execution")
print("=" * 55)
print()

# 2-qubit Grover: search for |11⟩
print("2-Qubit Grover (target: |11⟩):")
k2 = optimal_iterations(2, 1)
print(f"  Optimal iterations: {k2}")
counts2 = run_grover(2, [3], shots=2048)
for bs in sorted(counts2.keys()):
    pct = 100 * counts2[bs] / 2048
    bar = '█' * int(pct / 2)
    marker = ' ← target' if bs == '11' else ''
    print(f"  |{bs}⟩: {pct:5.1f}% {bar}{marker}")
print()

# 3-qubit Grover: search for |101⟩
print("3-Qubit Grover (target: |101⟩):")
k3 = optimal_iterations(3, 1)
print(f"  Optimal iterations: {k3}")
counts3 = run_grover(3, [5], shots=2048)
for bs in sorted(counts3.keys()):
    pct = 100 * counts3[bs] / 2048
    bar = '█' * int(pct / 2)
    marker = ' ← target' if bs == '101' else ''
    print(f"  |{bs}⟩: {pct:5.1f}% {bar}{marker}")
print()

# 4-qubit Grover: search for |1010⟩
print("4-Qubit Grover (target: |1010⟩):")
k4 = optimal_iterations(4, 1)
print(f"  Optimal iterations: {k4}")
counts4 = run_grover(4, [10], shots=2048)
target_count = counts4.get('1010', 0)
print(f"  P(target) = {100 * target_count / 2048:.1f}%")
print(f"  (total outcomes: {len(counts4)})")

---
## Section 4: Visualizing Amplitude Amplification

The heart of Grover's algorithm is **amplitude amplification**. After each iteration, the marked state's amplitude grows while others shrink.

The probability of success after $k$ iterations:

$$P(k) = \sin^2\left((2k+1)\theta\right), \quad \sin^2(\theta) = M/N$$

Let's visualize this rotation step by step.

In [ ]:
# ============================================================
# Section 4: Amplitude Amplification Visualization
# ============================================================

def amplitude_evolution(n_qubits, marked_states, max_iter):
    """Track amplitudes through Grover iterations."""
    N = 2 ** n_qubits
    oracle_mat = build_oracle_matrix(n_qubits, marked_states)
    diff_mat = build_diffusion_matrix(n_qubits)
    state = np.ones(N, dtype=complex) / np.sqrt(N)
    
    results = []
    marked_set = set(marked_states)
    for k in range(max_iter + 1):
        probs = np.abs(state) ** 2
        p_marked = sum(probs[i] for i in marked_set)
        results.append({
            'k': k,
            'p_marked': float(p_marked),
            'amplitudes': state.real.copy(),
            'probs': probs.copy()
        })
        if k < max_iter:
            state = oracle_mat @ state
            state = diff_mat @ state
    return results

# Track 3-qubit Grover evolution
print("📈 Amplitude Evolution: 3-qubit search for |101⟩")
print("=" * 55)
print()

evo = amplitude_evolution(3, [5], 6)
for step in evo:
    k = step['k']
    pm = step['p_marked']
    amp_target = step['amplitudes'][5]
    amp_other = step['amplitudes'][0]
    print(f"  k={k}: P(target)={pm:.4f}, amp(|101⟩)={amp_target:+.4f}, amp(others)={amp_other:+.4f}")
    
print()
print(f"  Optimal: k={optimal_iterations(3, 1)} → P={evo[2]['p_marked']:.4f}")
print(f"  Over-iterate: k=4 → P={evo[4]['p_marked']:.4f}  (worse!)")
print()
print("💡 Too many iterations = state rotates PAST the target.")
print("   This is 'Grover's soufflé': take it out at the right time!")

In [ ]:
# ============================================================
# Plotly Visualization: Amplitude Bar Chart Animation
# ============================================================

n_qubits_viz = 3
target_viz = 5  # |101⟩
max_k = 6
evo_data = amplitude_evolution(n_qubits_viz, [target_viz], max_k)

# Create subplots: amplitudes on left, probability on right
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Amplitudes per Iteration',
        'Success Probability vs Iteration',
        'Probability Distribution per Iteration',
        'Theoretical P(k) = sin²((2k+1)θ)'
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

N_viz = 2 ** n_qubits_viz
labels = [f'|{i:0{n_qubits_viz}b}⟩' for i in range(N_viz)]
colors = ['#636EFA' if i != target_viz else '#EF553B' for i in range(N_viz)]

# Top-left: amplitudes at each iteration
for step in evo_data:
    k = step['k']
    opacity = 0.3 + 0.7 * (k == optimal_iterations(n_qubits_viz, 1))
    fig.add_trace(
        go.Bar(x=labels, y=step['amplitudes'],
               name=f'k={k}', marker_color=colors,
               opacity=opacity,
               showlegend=(k == 0)),
        row=1, col=1
    )

# Show only optimal iteration in top-left
k_opt = optimal_iterations(n_qubits_viz, 1)
opt_step = evo_data[k_opt]
fig.data[k_opt].visible = True
for i, trace in enumerate(fig.data[:max_k + 1]):
    trace.visible = (i == k_opt)

# Top-right: success probability vs iteration
ks = [s['k'] for s in evo_data]
ps = [s['p_marked'] for s in evo_data]
fig.add_trace(
    go.Scatter(x=ks, y=ps, mode='lines+markers',
               name='P(success)',
               marker=dict(size=10, color='#EF553B'),
               line=dict(width=2)),
    row=1, col=2
)
fig.add_hline(y=1.0, line_dash='dash', line_color='gray',
              annotation_text='P=1', row=1, col=2)
fig.add_vline(x=k_opt, line_dash='dash', line_color='green',
              annotation_text=f'optimal k={k_opt}', row=1, col=2)

# Bottom-left: probability distribution at optimal
fig.add_trace(
    go.Bar(x=labels, y=opt_step['probs'],
           marker_color=colors, name='P at optimal k',
           showlegend=False),
    row=2, col=1
)

# Bottom-right: theoretical curve
theta = np.arcsin(np.sqrt(1 / N_viz))
k_fine = np.linspace(0, max_k, 200)
p_theory = np.sin((2 * k_fine + 1) * theta) ** 2
fig.add_trace(
    go.Scatter(x=k_fine, y=p_theory, mode='lines',
               name='Theory', line=dict(width=2, color='#AB63FA')),
    row=2, col=2
)
fig.add_trace(
    go.Scatter(x=ks, y=ps, mode='markers',
               name='Simulation', marker=dict(size=10, color='#EF553B')),
    row=2, col=2
)

fig.update_layout(
    height=700, width=1000,
    title_text=f'Grover\'s Algorithm: {n_qubits_viz}-Qubit Search for |{target_viz:0{n_qubits_viz}b}⟩',
    showlegend=False
)
fig.update_yaxes(title_text='Amplitude', row=1, col=1)
fig.update_yaxes(title_text='P(success)', row=1, col=2)
fig.update_yaxes(title_text='Probability', row=2, col=1)
fig.update_yaxes(title_text='P(success)', row=2, col=2)
fig.update_xaxes(title_text='Iteration k', row=1, col=2)
fig.update_xaxes(title_text='Iteration k', row=2, col=2)
fig.show()

---
## Section 5: Scaling — Classical vs Quantum

The fundamental comparison:

| Search Type | Query Complexity | 1M items | 1B items |
| :--- | :--- | :--- | :--- |
| Classical (unstructured) | $O(N)$ | 500,000 queries | 500,000,000 |
| Grover (quantum) | $O(\sqrt{N})$ | 785 queries | 24,836 |

This quadratic speedup is **provably optimal** — no quantum algorithm can do better for unstructured search (BBBV theorem, 1997).

In [ ]:
# ============================================================
# Section 5: Classical vs Quantum Scaling
# ============================================================

print("⚡ Classical vs Quantum Query Comparison")
print("=" * 55)
print()
print(f"{'Qubits':>6} {'N':>10} {'Classical':>12} {'Quantum':>10} {'Speedup':>10}")
print("-" * 55)

for n in [2, 4, 8, 10, 16, 20]:
    N = 2 ** n
    classical = N  # expected queries
    quantum = optimal_iterations(n, 1)
    speedup = classical / quantum
    print(f"{n:>6} {N:>10,} {classical:>12,} {quantum:>10,} {speedup:>10.1f}×")

print()
print("💡 At 20 qubits: 1,048,576 classical queries vs 804 quantum.")
print("   That's a 1,304× speedup — and it only grows with N.")

In [ ]:
# ============================================================
# Plotly: Scaling Comparison Chart
# ============================================================

qubit_range = np.arange(2, 22)
N_vals = 2 ** qubit_range
classical_queries = N_vals.astype(float)
quantum_queries = np.array([optimal_iterations(n, 1) for n in qubit_range], dtype=float)

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=qubit_range, y=classical_queries,
    name='Classical O(N)',
    mode='lines+markers',
    line=dict(width=3, color='#EF553B'),
    marker=dict(size=8)
))

fig2.add_trace(go.Scatter(
    x=qubit_range, y=quantum_queries,
    name='Grover O(√N)',
    mode='lines+markers',
    line=dict(width=3, color='#636EFA'),
    marker=dict(size=8)
))

fig2.update_layout(
    title='Classical vs Quantum Search: Query Complexity',
    xaxis_title='Number of Qubits (n)',
    yaxis_title='Number of Queries',
    yaxis_type='log',
    height=500, width=800,
    legend=dict(x=0.02, y=0.98)
)

fig2.show()

---
## Section 6: Multi-Solution Grover Search

When there are $M > 1$ marked states:
- Optimal iterations: $k \approx \frac{\pi}{4}\sqrt{N/M}$ — **fewer** iterations needed
- Each marked state gets roughly equal probability $\approx 1/M$
- Speedup: $O(N/M)$ classical → $O(\sqrt{N/M})$ quantum

In [ ]:
# ============================================================
# Section 6: Multi-Solution Grover
# ============================================================

print("🎯 Multi-Solution Grover Search")
print("=" * 55)
print()

# 3-qubit search with 2 targets: |001⟩ and |110⟩
targets = [1, 6]
target_labels = [f'|{t:03b}⟩' for t in targets]
n_iter = optimal_iterations(3, len(targets))

print(f"Targets: {', '.join(target_labels)}")
print(f"Optimal iterations: {n_iter}")
print()

counts_multi = run_grover(3, targets, shots=4096)
marked_set = {f'{t:03b}' for t in targets}
marked_total = sum(c for bs, c in counts_multi.items() if bs in marked_set)

print("Results:")
for bs in sorted(counts_multi.keys()):
    pct = 100 * counts_multi[bs] / 4096
    bar = '█' * int(pct / 2)
    marker = ' ← target' if bs in marked_set else ''
    print(f"  |{bs}⟩: {pct:5.1f}% {bar}{marker}")

print(f"\nTotal success rate: {100 * marked_total / 4096:.1f}%")
print()

# Effect of M on iteration count
print("Effect of number of solutions on iterations (4-qubit, N=16):")
for M in [1, 2, 4, 8]:
    k = optimal_iterations(4, M)
    theta = np.arcsin(np.sqrt(M / 16))
    p = np.sin((2 * k + 1) * theta) ** 2
    print(f"  M={M}: k={k}, P(success)={p:.4f}")

---
## Section 7: The Geometry — 2D Rotation Picture

Grover's algorithm is a rotation in the 2D plane spanned by:
- $|w\rangle$ = superposition of marked states
- $|w^\perp\rangle$ = superposition of unmarked states

The oracle reflects about $|w^\perp\rangle$, and the diffusion operator reflects about $|s\rangle$. Two reflections = one rotation by $2\theta$.

This geometric picture explains:
- Why $\sim\sqrt{N}$ iterations: you need to rotate by $\sim\pi/2$, and each step rotates by $2\theta \approx 2/\sqrt{N}$
- Why over-iteration fails: you rotate past $\pi/2$
- Why more solutions help: larger $\theta$ means bigger rotation per step

In [ ]:
# ============================================================
# Section 7: Geometric Visualization — Rotation in 2D
# ============================================================

# Visualize the rotation for 3-qubit Grover
n_viz = 3
N_viz = 2 ** n_viz
M_viz = 1
theta = np.arcsin(np.sqrt(M_viz / N_viz))
k_opt = optimal_iterations(n_viz, M_viz)

print("📐 Geometric Picture: Rotation in |w⟩-|w⊥⟩ Plane")
print("=" * 55)
print()
print(f"  θ = arcsin(√(1/{N_viz})) = {np.degrees(theta):.2f}°")
print(f"  Each iteration rotates by 2θ = {2*np.degrees(theta):.2f}°")
print(f"  Target angle: 90° (to land on |w⟩)")
print(f"  Iterations to reach ≈90°: {k_opt}")
print(f"  Actual angle after {k_opt} iterations: {np.degrees((2*k_opt+1)*theta):.2f}°")
print()

# Plotly: rotation visualization
fig3 = go.Figure()

# Draw the axes
fig3.add_trace(go.Scatter(
    x=[0, 1.2], y=[0, 0], mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='|w⊥⟩ axis', showlegend=True
))
fig3.add_trace(go.Scatter(
    x=[0, 0], y=[0, 1.2], mode='lines',
    line=dict(color='gray', width=1, dash='dash'),
    name='|w⟩ axis', showlegend=True
))

# Draw unit circle arc
t_arc = np.linspace(0, np.pi/2, 100)
fig3.add_trace(go.Scatter(
    x=np.cos(t_arc), y=np.sin(t_arc), mode='lines',
    line=dict(color='lightgray', width=1),
    showlegend=False
))

# Draw state vectors at each iteration
colors_rot = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3', '#FF6692']
for k in range(min(k_opt + 2, 7)):
    angle = (2 * k + 1) * theta
    x_k = np.cos(angle)
    y_k = np.sin(angle)
    
    color = colors_rot[k % len(colors_rot)]
    width = 3 if k == k_opt else 1.5
    
    # Arrow (state vector)
    fig3.add_annotation(
        x=x_k, y=y_k, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True,
        arrowhead=2, arrowsize=1.5, arrowwidth=width,
        arrowcolor=color
    )
    
    # Label
    p_k = np.sin(angle) ** 2
    fig3.add_trace(go.Scatter(
        x=[x_k * 1.1], y=[y_k * 1.1],
        text=[f'k={k}<br>P={p_k:.2f}'],
        mode='text',
        textfont=dict(size=10, color=color),
        showlegend=False
    ))

fig3.update_layout(
    title=f'Grover Rotation in |w⟩-|w⊥⟩ Plane ({n_viz} qubits)',
    xaxis=dict(title='⟨w⊥|ψ⟩', range=[-0.3, 1.4], scaleanchor='y'),
    yaxis=dict(title='⟨w|ψ⟩', range=[-0.3, 1.4]),
    height=600, width=600,
    showlegend=True
)

fig3.show()

---
## Key Takeaways

| Concept | Summary |
| :--- | :--- |
| **Oracle** | Phase-flips marked states: $O = I - 2\vert w\rangle\langle w\vert$ |
| **Diffusion** | Reflects about mean: $D = 2\vert s\rangle\langle s\vert - I$ |
| **Geometry** | Two reflections = rotation by $2\theta$ in 2D subspace |
| **Optimal iterations** | $k \approx \frac{\pi}{4}\sqrt{N/M}$ |
| **Speedup** | Quadratic: $O(\sqrt{N})$ vs classical $O(N)$ |
| **Over-iteration** | Probability oscillates — must stop at optimal $k$ |
| **Multi-solution** | More targets → fewer iterations, same speedup ratio |

**Next:** Day 7 — Quantum Fourier Transform, the mathematical heart of Shor's algorithm.